# Experiment 01.01 — BTW Sandpile SOC Signature Validation

Validate diagnostic functions against the Bak-Tang-Wiesenfeld sandpile, the canonical SOC system. The goal is to confirm that our signature detection tools correctly identify SOC properties in a system where criticality is guaranteed by construction.

**Reference design:** [`work/experiments/validation/01_sandpile.md`](experiments/validation/01_sandpile.md)

**Implementation:** [`work/experiments/validation/sandpile.jl`](experiments/validation/sandpile.jl), [`work/experiments/validation/diagnostics.jl`](experiments/validation/diagnostics.jl)

## Setup

In [1]:
include("experiments/validation/load_validation.jl")

using DataFrames
using Statistics
using Printf

## 1. Smoke test — small lattice

Run a tiny BTW sandpile to verify the simulator works end-to-end before doing anything expensive.

In [2]:
smoke = btw_sandpile(32; N_transient=10_000, N_record=10_000, seed=1)

smoke_sizes = avalanche_sizes(smoke.catalog)

println("Smoke test — L=32, N_record=10,000:")
@printf("  Mean lattice height: %.3f (expected ~2.125 for BTW)\n", mean(smoke.final_height))
@printf("  Non-empty avalanches: %d / %d\n", length(smoke_sizes), length(smoke.catalog))
@printf("  Mean avalanche size:  %.2f\n", mean(smoke_sizes))
@printf("  Max avalanche size:   %d\n", maximum(smoke_sizes))

Smoke test — L=32, N_record=10,000:
  Mean lattice height: 2.080 (expected ~2.125 for BTW)
  Non-empty avalanches: 4216 / 10000
  Mean avalanche size:  95.46
  Max avalanche size:   2137


## 1b. Burn-in convergence trace

With `:empty` initial condition, there is no clean "now at criticality" signal — only convergence diagnostics. We monitor:

- **`mean_z`** should climb monotonically and plateau at **17/8 = 2.125** (Dhar 1999, exact stationary value for 2D BTW).
- **`dissipation_rate_recent`** should rise and plateau at **1.0** (one grain leaves per grain added at steady state).
- **`mean_avalanche_size_recent`** should stabilize.

Run once per lattice size to choose an appropriate `N_transient` for production runs.

In [3]:
trace_L = 64
trace = btw_burnin_trace(trace_L;
                         N_grains=10 * trace_L * trace_L,
                         log_every=1000,
                         seed=1)
show(DataFrame(trace), allrows=true, allcols=true)

40×5 DataFrame
 Row │ step   mean_z    cumulative_dissipation  dissipation_rate_recent  mean_avalanche_size_recent 
     │ Int64  Float64   Int64                   Float64                  Float64                    
─────┼──────────────────────────────────────────────────────────────────────────────────────────────
   1 │  1000  0.244141                       0                    0.0                         0.001
   2 │  2000  0.488281                       0                    0.0                         0.004
   3 │  3000  0.732422                       0                    0.0                         0.025
   4 │  4000  0.976074                       2                    0.002                       0.061
   5 │  5000  1.21802                       11                    0.009                       0.148
   6 │  6000  1.45874                       25                    0.014                       0.268
   7 │  7000  1.69775                       46                    0.021           

**Reading the table:** find the step where `mean_z` first exceeds ~2.10 and stays there, and where `dissipation_rate_recent` is consistently near 1.0. That step is the empirical N_transient for this lattice size. Round up for safety. The default `10 × L²` used below is conservative.

In [4]:
# Optional: same trace from :overloaded initial condition for comparison.
# Should converge much faster (the initial cascade does most of the work).
trace_overloaded = btw_burnin_trace(trace_L;
                                    N_grains=2 * trace_L * trace_L,
                                    log_every=1000,
                                    initial_condition=:overloaded,
                                    seed=1)
DataFrame(trace_overloaded)

Row,step,mean_z,cumulative_dissipation,dissipation_rate_recent,mean_avalanche_size_recent
,Int64,Float64,Int64,Float64,Float64
1,1000,2.11182,988,0.988,152.727
2,2000,2.10107,2032,1.044,156.843
3,3000,2.10107,3032,1.0,157.67
4,4000,2.09644,4051,1.019,152.64
5,5000,2.10083,5033,0.982,154.093
6,6000,2.10327,6023,0.99,149.048
7,7000,2.09595,7053,1.03,154.394
8,8000,2.12036,7953,0.9,138.868


## 1c. Adaptive burn-in

Rather than hardcoding `N_transient` as a multiple of L², detect steady state from the observed signals: `mean_z` plateau + `dissipation_rate` near 1.0. This works for any L and any initial condition.

Convergence criterion (default):
- Range of `mean_z` over last 5 check windows < 0.02
- Mean of `dissipation_rate_recent` over last 5 windows in [0.9, 1.1]

Returns the burn-in trace so you can see when and how convergence happened.

In [5]:
# Empty start — should converge around step ~10K for L=64
adaptive_empty = btw_sandpile_adaptive(64;
                                       N_record=50_000,
                                       initial_condition=:empty,
                                       seed=42)
@printf("Empty start:      converged=%s  burnin=%d grains\n",
        adaptive_empty.converged, adaptive_empty.n_burnin_grains)

# Overloaded start — should converge immediately
adaptive_over = btw_sandpile_adaptive(64;
                                     N_record=50_000,
                                     initial_condition=:overloaded,
                                     seed=42)
@printf("Overloaded start: converged=%s  burnin=%d grains\n",
        adaptive_over.converged, adaptive_over.n_burnin_grains)

Empty start:      converged=true  burnin=14000 grains
Overloaded start: converged=true  burnin=5000 grains


In [6]:
# Inspect the burn-in trace to see the convergence path
show(DataFrame(adaptive_empty.burnin_trace), allrows=true, allcols=true)

14×5 DataFrame
 Row │ step   mean_z    cumulative_dissipation  dissipation_rate_recent  mean_avalanche_size_recent 
     │ Int64  Float64   Int64                   Float64                  Float64                    
─────┼──────────────────────────────────────────────────────────────────────────────────────────────
   1 │  1000  0.244141                       0                    0.0                         0.001
   2 │  2000  0.488281                       0                    0.0                         0.005
   3 │  3000  0.732178                       1                    0.001                       0.027
   4 │  4000  0.974854                       7                    0.006                       0.08
   5 │  5000  1.21729                       14                    0.007                       0.152
   6 │  6000  1.45728                       31                    0.017                       0.337
   7 │  7000  1.69702                       49                    0.018            

## 2. Production runs at L = 64, 128, 256

Uses `btw_sandpile_adaptive` — burn-in length determined by convergence detection (section 1c) rather than a hardcoded multiplier of L².

Starting with L=64 and L=128; rerun at L=256 once tools are confirmed working.

In [7]:
# Testing :overloaded initial condition to verify that the Clauset
# fit result is not a burn-in artifact. If :empty and :overloaded
# give the same alpha / LLR, the weird result is intrinsic to BTW.
L_values = [64, 128]
N_record = 200_000

results = Dict{Int, NamedTuple}()

for L in L_values
    @printf("L=%d: record=%d ... ", L, N_record)
    t = @elapsed res = btw_sandpile_adaptive(L;
                           N_record=N_record,
                           initial_condition=:overloaded,
                           seed=42)
    results[L] = res
    @printf("done in %.1fs  burnin=%d grains  converged=%s  mean_height=%.3f\n",
            t, res.n_burnin_grains, res.converged,
            mean(res.final_height))
    if !res.converged
        @warn "L=$L did not converge — results may be non-stationary"
    end
end

L=64: record=200000 ... done in 2.7s  burnin=5000 grains  converged=true  mean_height=2.095
L=128: record=200000 ... done in 10.4s  burnin=5000 grains  converged=true  mean_height=2.117


### Catalog summary

In [8]:
summary_rows = NamedTuple[]
for L in L_values
    sizes = avalanche_sizes(results[L].catalog)
    durations = avalanche_durations(results[L].catalog)
    push!(summary_rows, (
        L = L,
        n_recorded = length(results[L].catalog),
        n_nonempty = length(sizes),
        mean_size = mean(sizes),
        median_size = median(sizes),
        max_size = maximum(sizes),
        mean_duration = mean(durations),
        max_duration = maximum(durations),
        mean_height = mean(results[L].final_height),
    ))
end
DataFrame(summary_rows)

Row,L,n_recorded,n_nonempty,mean_size,median_size,max_size,mean_duration,max_duration,mean_height
,Int64,Int64,Int64,Float64,Float64,Int64,Float64,Int64,Float64
1,64,200000,87425,349.381,28.0,17413,39.3098,831,2.09497
2,128,200000,88694,1335.65,42.0,122657,79.3994,2764,2.11694


## 3. Signature 1 — Power-law avalanche size distribution

Expected: tau_s ≈ 1.2 (effective; BTW exhibits multiscaling per Tebaldi et al. 1999, so a single exponent is approximate).

In [9]:
pl_rows = NamedTuple[]
for L in L_values
    sizes = avalanche_sizes(results[L].catalog)
    fit = fit_power_law(Float64.(sizes))
    cmp = compare_power_law_exponential(Float64.(sizes), fit.xmin, fit.alpha)
    push!(pl_rows, (
        L = L,
        xmin = fit.xmin,
        alpha = fit.alpha,
        sigma_alpha = fit.sigma_alpha,
        ks_distance = fit.ks_distance,
        n_tail = fit.n_tail,
        ll_ratio_pl_vs_exp = cmp.log_likelihood_ratio,
    ))
end
DataFrame(pl_rows)

Row,L,xmin,alpha,sigma_alpha,ks_distance,n_tail,ll_ratio_pl_vs_exp
,Int64,Float64,Float64,Float64,Float64,Int64,Float64
1,64,1736.0,2.84083,0.0264874,0.0615623,4830,-127.357
2,128,7111.0,2.52401,0.0227237,0.0545253,4498,-19.3448


### 3a. Visualize P(s) to identify the scaling regime

Clauset's auto-xmin search landed in the finite-size cutoff region. Plotting log-binned P(s) on log-log reveals the three regimes:

- **Small s** (< 10ish): deviations due to lattice discretization
- **Middle s** (~10-1000 for L=64): the scaling regime where tau_s ≈ 1.2 lives
- **Large s** (upper tail): finite-size cutoff where the distribution curves down

In [ ]:
using Plots

p = plot(xaxis=:log, yaxis=:log,
         xlabel="avalanche size s",
         ylabel="P(s)",
         title="BTW avalanche size distribution",
         legend=:topright)

for L in L_values
    sizes = avalanche_sizes(results[L].catalog)
    h = log_binned_pmf(Float64.(sizes); n_bins=50)
    plot!(p, h.bin_centers, h.pmf, label="L=$L", marker=:circle, markersize=3)
end

p

### 3b. Manual xmin in the scaling regime

Based on the plot, fit with xmin in the middle decade (not the top 5% the auto-search found). Try a grid of xmin values to see how tau_s varies — this also makes the multiscaling visible (a true power law would give constant alpha across xmin choices).

In [ ]:
manual_xmins = [5, 10, 30, 100, 300]

rows = NamedTuple[]
for L in L_values
    sizes = Float64.(avalanche_sizes(results[L].catalog))
    for xm in manual_xmins
        fit = fit_power_law(sizes; xmin=xm)
        cmp = compare_power_law_exponential(sizes, xm, fit.alpha)
        push!(rows, (
            L = L,
            xmin = xm,
            alpha = fit.alpha,
            sigma_alpha = fit.sigma_alpha,
            ks_distance = fit.ks_distance,
            n_tail = fit.n_tail,
            ll_ratio_pl_vs_exp = cmp.log_likelihood_ratio,
        ))
    end
end
DataFrame(rows)

### 3c. Fit the scaling regime only (xmin + xmax)

To isolate the scaling regime from the cutoff, provide both xmin and xmax. The upper xmax should exclude the region where P(s) curves away from power-law behavior. For L=64, that's roughly s < 1000 (below the visible knee in 3a).

In [ ]:
# Scaling-regime fits: xmin at 10, xmax truncates the cutoff.
# Expected: tau_s closer to 1.2 (published BTW value)
regime_rows = NamedTuple[]
for (L, xmax_L) in [(64, 1000.0), (128, 4000.0)]
    sizes = Float64.(avalanche_sizes(results[L].catalog))
    fit = fit_power_law(sizes; xmin=10.0, xmax=xmax_L)
    push!(regime_rows, (
        L = L,
        xmin = 10.0,
        xmax = xmax_L,
        alpha_scaling_regime = fit.alpha,
        sigma_alpha = fit.sigma_alpha,
        ks_distance = fit.ks_distance,
        n_tail = fit.n_tail,
    ))
end
DataFrame(regime_rows)

### 3d. Area distribution — cleaner scaling than size

Tebaldi, De Menech, Stella (1999) showed the BTW avalanche *area* (distinct sites toppled) obeys clean simple finite-size scaling — unlike the size distribution which has multiscaling. Expected tau_a ≈ 1.37.

In [ ]:
p_area = plot(xaxis=:log, yaxis=:log,
              xlabel="avalanche area a",
              ylabel="P(a)",
              title="BTW avalanche area distribution",
              legend=:topright)

area_rows = NamedTuple[]
for L in L_values
    areas = Float64.(avalanche_areas(results[L].catalog))
    h = log_binned_pmf(areas; n_bins=50)
    plot!(p_area, h.bin_centers, h.pmf, label="L=$L",
          marker=:circle, markersize=3)

    fit = fit_power_law(areas)  # auto xmin — should work better for area
    push!(area_rows, (
        L = L,
        xmin = fit.xmin,
        alpha_area = fit.alpha,
        sigma_alpha = fit.sigma_alpha,
        ks_distance = fit.ks_distance,
        n_tail = fit.n_tail,
    ))
end
p_area

In [ ]:
DataFrame(area_rows)

Same fit for duration distribution (tau_t expected ~1.4-1.5):

In [10]:
dur_rows = NamedTuple[]
for L in L_values
    durs = avalanche_durations(results[L].catalog)
    fit = fit_power_law(Float64.(durs))
    push!(dur_rows, (
        L = L,
        xmin = fit.xmin,
        alpha_duration = fit.alpha,
        sigma_alpha = fit.sigma_alpha,
        ks_distance = fit.ks_distance,
        n_tail = fit.n_tail,
    ))
end
DataFrame(dur_rows)

Row,L,xmin,alpha_duration,sigma_alpha,ks_distance,n_tail
,Int64,Float64,Float64,Float64,Float64,Int64
1,64,170.0,4.28871,0.0495173,0.0672631,4411
2,128,386.0,3.76074,0.040892,0.0536965,4558


## 4. Signature 2 — Spatial correlation in the height field

G(r) should decay as a power law (with logarithmic corrections in BTW), not exponentially. Expensive at large L; sample if needed.

In [11]:
# Only compute for L=64 (correlation_function is O(L^4))
corr64 = correlation_function(results[64].final_height; max_r=16)
println("Height-height correlation G(r) at L=64:")
for (r, g, n) in zip(corr64.r, corr64.G, corr64.n_pairs)
    @printf("  r=%2d  G(r)=% .4f  (n_pairs=%d)\n", r, g, n)
end

Height-height correlation G(r) at L=64:
  r= 1  G(r)=-0.0517  (n_pairs=32004)
  r= 2  G(r)= 0.0118  (n_pairs=47120)
  r= 3  G(r)= 0.0189  (n_pairs=61736)
  r= 4  G(r)= 0.0252  (n_pairs=120500)
  r= 5  G(r)= 0.0272  (n_pairs=103384)
  r= 6  G(r)= 0.0335  (n_pairs=144360)
  r= 7  G(r)= 0.0315  (n_pairs=141660)
  r= 8  G(r)= 0.0301  (n_pairs=166344)
  r= 9  G(r)= 0.0366  (n_pairs=229912)
  r=10  G(r)= 0.0233  (n_pairs=185516)
  r=11  G(r)= 0.0225  (n_pairs=232520)
  r=12  G(r)= 0.0230  (n_pairs=215464)
  r=13  G(r)= 0.0281  (n_pairs=271388)
  r=14  G(r)= 0.0225  (n_pairs=265152)
  r=15  G(r)= 0.0271  (n_pairs=248136)
  r=16  G(r)= 0.0188  (n_pairs=321556)


## 5. Signature 3 — Spectral analysis

Compute PSD of the avalanche size time series. **Note:** BTW does NOT produce clean 1/f noise. Expected high-frequency exponent ~1.56 with three distinct frequency regimes (Chhimpa et al. 2025).

In [12]:
spec_rows = NamedTuple[]
for L in L_values
    # Use full sequence including empty avalanches (preserves time structure)
    series = avalanche_sizes(results[L].catalog; exclude_empty=false)
    spec = power_spectrum(Float64.(series))
    beta_fit = spectral_exponent(spec.freq, spec.psd)
    h = hurst_rs(Float64.(series))
    push!(spec_rows, (
        L = L,
        beta = beta_fit.beta,
        beta_n_used = beta_fit.n_used,
        hurst = h.H,
        hurst_n_scales = length(h.scales),
    ))
end
DataFrame(spec_rows)

Row,L,beta,beta_n_used,hurst,hurst_n_scales
,Int64,Float64,Int64,Float64,Int64
1,64,-0.00208233,79999,0.358256,20
2,128,-0.0024709,79999,0.406126,20


## 6. Signature 4 — Fractal structure of avalanche frontiers

BTW avalanche clusters are approximately compact (D ≈ 2). The fractal property is in the **frontier** (boundary), expected D_frontier ≈ 1.25 (Moghimi-Araghi et al. 2009).

For now we measure the cluster footprint dimension; frontier extraction can be added later.

In [13]:
# Re-run a small set of avalanches with site tracking would require modifying
# the simulator. The current AvalancheRecord only stores aggregate stats.
# For now: report the size-area scaling exponent as a proxy.
#
# s ~ a^gamma where gamma > 1 indicates multiple topplings per site (BTW signature)

fractal_rows = NamedTuple[]
for L in L_values
    catalog = results[L].catalog
    nonempty = [a for a in catalog if a.size > 0]
    sizes = Float64.([a.size for a in nonempty])
    areas = Float64.([a.area for a in nonempty])
    # Fit log(size) vs log(area) on the upper half (large avalanches)
    big_mask = sizes .>= quantile(sizes, 0.5)
    log_s = log.(sizes[big_mask])
    log_a = log.(areas[big_mask])
    mean_x = mean(log_a)
    mean_y = mean(log_s)
    gamma = sum((log_a .- mean_x) .* (log_s .- mean_y)) / sum((log_a .- mean_x).^2)
    push!(fractal_rows, (
        L = L,
        size_area_exponent_gamma = gamma,
        n_used = sum(big_mask),
    ))
end
DataFrame(fractal_rows)

Row,L,size_area_exponent_gamma,n_used
,Int64,Float64,Int64
1,64,1.08359,43819
2,128,1.09019,44333


## 7. Signature 5 — Fat-tailed changes

Excess kurtosis of first differences in rolling avalanche activity. Expected: K > 1.5.

In [14]:
kurt_rows = NamedTuple[]
for L in L_values
    series = avalanche_sizes(results[L].catalog; exclude_empty=false)
    raw = fat_tail_kurtosis(Float64.(series); window=1)
    win100 = fat_tail_kurtosis(Float64.(series); window=100)
    win1000 = fat_tail_kurtosis(Float64.(series); window=1000)
    push!(kurt_rows, (
        L = L,
        excess_kurtosis_raw = raw.excess_kurtosis,
        excess_kurtosis_w100 = win100.excess_kurtosis,
        excess_kurtosis_w1000 = win1000.excess_kurtosis,
    ))
end
DataFrame(kurt_rows)

Row,L,excess_kurtosis_raw,excess_kurtosis_w100,excess_kurtosis_w1000
,Int64,Float64,Float64,Float64
1,64,44.896,44.9917,45.2789
2,128,88.2416,88.7526,88.1271


## 8. Branching ratio (activity-dependent)

Per Michiels van Kessenich et al. (2010): a single global b is misleading. Look for a **broad activity range where b(x) ≈ 1** — that is the SOC signature.

In [15]:
for L in L_values
    bx = branching_ratio_activity_dependent(results[L].catalog; n_bins=20)
    global_b = branching_ratio_global(results[L].catalog)
    println("L=$L:")
    @printf("  Global mean b: %.3f\n", global_b)
    println("  Activity-dependent b(x):")
    for (a, b, n) in zip(bx.activity_levels, bx.b_of_x, bx.n_in_bin)
        @printf("    activity ~ %7.1f   b(x) = %.3f   (n=%d)\n", a, b, n)
    end
    println()
end

L=64:
  Global mean b: 1.071
  Activity-dependent b(x):
    activity ~     1.1   b(x) = 1.592   (n=272806)
    activity ~     1.8   b(x) = 1.168   (n=311009)
    activity ~     3.0   b(x) = 1.076   (n=309512)
    activity ~     3.8   b(x) = 1.040   (n=281652)
    activity ~     4.8   b(x) = 1.022   (n=252308)
    activity ~     6.1   b(x) = 1.011   (n=224297)
    activity ~     7.8   b(x) = 1.002   (n=374858)
    activity ~     9.9   b(x) = 0.992   (n=411396)
    activity ~    12.6   b(x) = 0.986   (n=279492)
    activity ~    16.1   b(x) = 0.982   (n=239856)
    activity ~    20.4   b(x) = 0.979   (n=172897)
    activity ~    26.0   b(x) = 0.978   (n=107262)
    activity ~    33.1   b(x) = 0.977   (n=65551)
    activity ~    42.2   b(x) = 0.974   (n=29178)
    activity ~    53.7   b(x) = 0.971   (n=10053)
    activity ~    68.4   b(x) = 0.972   (n=2088)
    activity ~    87.0   b(x) = 0.966   (n=315)
    activity ~   110.8   b(x) = 0.971   (n=26)

L=128:
  Global mean b: 1.046
  Activ

## 9. Inter-event time distribution

Waiting times between avalanches above selected thresholds. Expected: power-law or stretched exponential at high thresholds, no characteristic waiting time.

In [16]:
for L in L_values
    series = avalanche_sizes(results[L].catalog; exclude_empty=false)
    nonempty = avalanche_sizes(results[L].catalog)
    println("L=$L thresholds and waiting time stats:")
    for q in [0.50, 0.90, 0.99]
        thr = quantile(Float64.(nonempty), q)
        waits = inter_event_times(Float64.(series), thr)
        if length(waits) >= 10
            @printf("  q=%.2f  thr=%.0f  n_waits=%d  mean=%.1f  median=%.1f  max=%d\n",
                    q, thr, length(waits), mean(waits), median(waits), maximum(waits))
        else
            @printf("  q=%.2f  thr=%.0f  n_waits=%d  (insufficient)\n",
                    q, thr, length(waits))
        end
    end
    println()
end

L=64 thresholds and waiting time stats:
  q=0.50  thr=29  n_waits=43818  mean=4.6  median=3.0  max=54
  q=0.90  thr=1023  n_waits=8717  mean=22.9  median=17.0  max=153
  q=0.99  thr=4509  n_waits=873  mean=228.7  median=187.0  max=1180

L=128 thresholds and waiting time stats:
  q=0.50  thr=41  n_waits=44332  mean=4.5  median=3.0  max=54
  q=0.90  thr=3371  n_waits=8829  mean=22.6  median=16.0  max=182
  q=0.99  thr=22071  n_waits=882  mean=226.3  median=173.0  max=1194



## 10. Summary against published values

Compare what we measured to literature for 2D BTW. See [`01_sandpile.md`](experiments/validation/01_sandpile.md) §References for sources.

| Quantity | Expected | Status |
|----------|----------|--------|
| Mean height | 2.125 (exact, Dhar 1999) | Compare to `mean_height` above |
| tau_s (size) | ~1.2 effective; multiscaling | Compare to `alpha` in Section 3 |
| tau_t (duration) | ~1.4-1.5 | Compare to `alpha_duration` in Section 3 |
| PSD beta | ~1.56 high-freq (NOT 1/f) | Compare to `beta` in Section 5 |
| Hurst H | > 0.5 (persistent) | Compare to `hurst` in Section 5 |
| Excess kurtosis | > 1.5 | Compare in Section 7 |
| Branching b(x) | broad regime ≈ 1 | Compare in Section 8 |

**Next steps when this passes:**
- Increase to L=256 and N_record=10⁶ for production exponents
- Add negative controls (Poisson, subcritical, supercritical)
- Implement frontier extraction for true fractal dimension measurement
- Multiscaling test: moment ratios across L values
- Add Manna model for universality-class comparison (deferred per Experiment 01.02)